In [1]:
import os
from dotenv import load_dotenv

from typing import TypedDict

# =====================================================
# LOAD ENVIRONMENT VARIABLES
# =====================================================

load_dotenv("../.env",override=True)

True

In [24]:
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")

In [25]:

# =====================================================
# LOAD CSV
# =====================================================

from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path="customers.csv"
)

documents = loader.load()

In [26]:
print(documents[0].page_content)

customer_id: 1
name: Saravana
loan_amount: 500000
city: Chennai


In [27]:
# =====================================================
# SPLIT DOCUMENTS
# =====================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

In [28]:
chunks

[Document(metadata={'source': 'customers.csv', 'row': 0}, page_content='customer_id: 1\nname: Saravana\nloan_amount: 500000\ncity: Chennai'),
 Document(metadata={'source': 'customers.csv', 'row': 1}, page_content='customer_id: 2\nname: Ravi\nloan_amount: 300000\ncity: Madurai'),
 Document(metadata={'source': 'customers.csv', 'row': 2}, page_content='customer_id: 3\nname: John\nloan_amount: 200000\ncity: Bangalore'),
 Document(metadata={'source': 'customers.csv', 'row': 3}, page_content='customer_id: 4\nname: Kumar\nloan_amount: 800000\ncity: Hyderabad')]

In [29]:


# =====================================================
# EMBEDDINGS
# =====================================================

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2"
)


In [ ]:

# =====================================================
# VECTOR STORE
# =====================================================

from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

# =====================================================
# GEMINI MODEL
# =====================================================

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [20]:
# =====================================================
# STATE DEFINITION
# =====================================================

class GraphState(TypedDict):
    question: str
    context: str
    answer: str

In [21]:
# =====================================================
# RETRIEVE NODE
# =====================================================

def retrieve(state: GraphState):

    question = state["question"]

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    return {
        "context": context
    }

In [22]:


# =====================================================
# GENERATE NODE
# =====================================================

def generate(state: GraphState):

    question = state["question"]
    context = state["context"]

    prompt = f"""
You are an intelligent assistant.

Answer ONLY from the provided context.

If the answer is not available in the context,
respond with:
"I could not find the answer in the provided data."

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }


In [23]:

# =====================================================
# BUILD LANGGRAPH
# =====================================================

from langgraph.graph import StateGraph, START, END

builder = StateGraph(GraphState)

builder.add_node(
    "retrieve",
    retrieve
)

builder.add_node(
    "generate",
    generate
)

builder.add_edge(
    START,
    "retrieve"
)

builder.add_edge(
    "retrieve",
    "generate"
)

builder.add_edge(
    "generate",
    END
)

graph = builder.compile()

In [ ]:
print("\nCSV RAG Chatbot Started")
print("Type 'exit' to quit\n")

while True:

    question = input("Ask Question: ")

    if question.lower() == "exit":
        break

    result = graph.invoke(
        {
            "question": question
        }
    )

    print("\nAnswer:")
    print(result["answer"])
    print("-" * 60)